# ZS601 D：C + LiDAR贴面受控增密

D从同一LiDAR重新初始化，继承C的几何loss和硬尺度上限，只新增独立的 surface_densify。
子高斯沿可靠LiDAR局部平面切向产生并投影回平面；标准densify_and_prune保持关闭。
150000步、Tesla L4、SH2、seed42。数据先复制到/content；checkpoint每50000步。
正式完成条件包括：全test指标、masked PSNR最差10相机的RGB/1σ椭球/深度/法向、experiment_summary.md。


## 1. 挂载Drive并检查Tesla L4


In [ ]:
from google.colab import drive
drive.mount('/content/drive', timeout_ms=300000)


In [ ]:
from pathlib import Path
import sys,subprocess,uuid,json,shutil,zipfile,csv,os
import numpy as np
import torch
CODE_REF='dc239ced4437fc67eddbec6434f57435371fe61b'
REPO_URL='https://github.com/VISjudy/ZS601_3DGS.git'
DATA_ZIP=Path('/content/drive/MyDrive/LCCDataset/ZS601meetingroom/ZS601meetingroom_data.zip')
HISTORY=Path('/content/drive/MyDrive/LCCDataset/zs601_output/gaussian-splattingWithMask_v3_cff221ccfb')
VAL_SOURCE=HISTORY/'images-val10.txt'
TEST_SOURCE=HISTORY/'images_test.txt'
BASELINE_RESULT=None # 设为完成同协议final test的A输出目录后自动计算数值差
assert os.path.ismount('/content/drive')
assert DATA_ZIP.is_file() and VAL_SOURCE.is_file() and TEST_SOURCE.is_file()
WORK=Path('/content')/('zs601_D_'+uuid.uuid4().hex[:10]);WORK.mkdir()
RESULTS=HISTORY.parent/WORK.name;RESULTS.mkdir()
ITERATIONS=150000
COMMON={'sh_degree':2,'position_lr_init':0.000016,'position_lr_final':0.00000016,
        'position_lr_max_steps':150000,'scaling_lr':0.0015,'seed':42,'lazy_cache':100}
OVERRIDES={} # 例：{'surface_densify':'off'} 应退化为C
def run(args,cwd=None):
    args=list(map(str,args));print('COMMAND',args,flush=True)
    logfile=RESULTS/('command_'+uuid.uuid4().hex[:10]+'.log')
    with logfile.open('x') as log,subprocess.Popen(args,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True) as p:
        for line in p.stdout:
            log.write(line)
            if not line.startswith('Reading camera'): print(line,end='',flush=True)
        if p.wait(): raise subprocess.CalledProcessError(p.returncode,args)
    print('LOG',logfile)
assert torch.cuda.is_available() and 'L4' in torch.cuda.get_device_name(0)
run(['nvidia-smi'])
run(['git','clone','--branch','v3-d',REPO_URL,WORK/'repo'])
run(['git','checkout','--detach',CODE_REF],cwd=WORK/'repo')
CODE=WORK/'repo/gaussian-splattingWithMask_v3'
print({'python':sys.version,'torch':torch.__version__,'cuda':torch.version.cuda,'code':CODE_REF})


## 2. 构建CUDA扩展并运行单元测试


In [ ]:
assert torch.version.cuda and torch.version.cuda.split('.')[0]=='12'
run([sys.executable,'-m','pip','install','ninja','plyfile','laspy','scipy','pillow','opencv-python-headless','tqdm','cupy-cuda12x'])
import cupy as cp
assert cp.cuda.runtime.getDeviceCount()>0 and int(cp.arange(10).sum().get())==45
os.environ['MAX_JOBS']='2'
for name in ['simple-knn','diff-gaussian-rasterization']:
    run([sys.executable,'-m','pip','install','-v','--no-build-isolation',CODE/'submodules'/name])
run([sys.executable,'-m','unittest','-v','test_geometry_v3','test_scale_bounds_v3','test_surface_densify_v3'],cwd=CODE)


## 3. 将数据从Drive复制到Colab本地


In [ ]:
LOCAL_ZIP=WORK/'data.zip';shutil.copy2(DATA_ZIP,LOCAL_ZIP)
DATA=WORK/'data';DATA.mkdir()
with zipfile.ZipFile(LOCAL_ZIP) as z:
    for item in z.infolist():
        target=(DATA/item.filename).resolve()
        assert target.is_relative_to(DATA.resolve()),item.filename
        assert (item.external_attr>>16)&0o170000!=0o120000,'ZIP symlink'
    z.extractall(DATA)
POINTS=DATA/'ZS601_3cm_sample.las'
INTR=DATA/'sparse/cameras.txt';POSES=DATA/'sparse/images.txt'
VAL=WORK/'images-val10.txt';TEST=WORK/'images_test.txt';TRAIN=WORK/'images_train_v3.txt'
shutil.copy2(VAL_SOURCE,VAL);shutil.copy2(TEST_SOURCE,TEST)
for p in [POINTS,INTR,POSES,VAL,TEST]: assert p.is_file(),str(p)
run([sys.executable,'prepare_v3.py','--images_file',POSES,'--val_file',VAL,'--test_file',TEST,'--output_train',TRAIN],cwd=CODE)
for p in [VAL,TEST,TRAIN]: shutil.copy2(p,RESULTS/p.name)
print('LOCAL INPUTS',DATA,POINTS,INTR,TRAIN,VAL,TEST)


In [ ]:
def train_command(out,iterations,final_test):
    cmd=[sys.executable,'train_mask_v3.py','--experiment','D','-s',DATA,'-m',out,
         '--point_cloud',POINTS,'--train_file',TRAIN,'--val_file',VAL,'--test_file',TEST,
         '--cameras_file',INTR,'--units','scene','--iterations',iterations,
         '--val_interval',5000,'--checkpoint_interval',50000,'--val_npz','off',
         '--val_ellipsoids','on','--final_test',final_test]
    if BASELINE_RESULT: cmd+=['--baseline_result',BASELINE_RESULT]
    for key,value in {**COMMON,**OVERRIDES}.items(): cmd+=['--'+key,str(value)]
    return cmd

def verify(out,steps,formal):
    done=json.loads((out/'completed.json').read_text());assert done['iteration']==steps
    loss=list(csv.DictReader((out/'loss_log.csv').open()))
    assert len(loss)==steps and [int(r['iteration']) for r in loss]==list(range(1,steps+1))
    assert all(np.isfinite(float(v)) for r in loss for v in r.values())
    cfg=json.loads((out/'run_config.json').read_text())
    assert cfg['experiment']=='D' and cfg['surface_densify'] and cfg['scale_bounds']
    assert sorted(p.name for p in (out/'checkpoints').glob('*.pth'))==sorted(
        f'iteration_{i}.pth' for i in sorted(set(range(50000,steps+1,50000))|{steps}))
    if formal:
        assert done['final_test_complete'] is True
        test_dir=out/'test_final/iteration_150000'
        metrics=list(csv.DictReader((test_dir/'test_metrics.csv').open()))
        manifest=json.loads((out/'run_manifest.json').read_text())
        assert len(metrics)==len(manifest['test_names'])+4
        for kind in ['rgb','ellipsoid','depth','normal']:
            assert len(list(test_dir.glob('*_'+kind+'.png')))==10,(kind,test_dir)
        assert not list(test_dir.glob('*.npz'))
        assert (test_dir/'test_summary.json').is_file()
        assert (out/'experiment_summary.md').is_file()
    else:
        assert done['final_test_complete'] is False
    print('VERIFIED',out,done,'loss rows',len(loss))
    return {'path':str(out),**done,'loss_rows':len(loss)}


## 4. 200步D冒烟


In [ ]:
SMOKE_OUT=RESULTS/'smoke_D'
run(train_command(SMOKE_OUT,200,'off'),cwd=CODE)
SMOKE=verify(SMOKE_OUT,200,False)
from IPython.display import display
from PIL import Image
for name in ['val00','val03']:
    for kind in ['rgb','ellipsoid']:
        display(Image.open(SMOKE_OUT/'val_v3/iteration_000200'/f'{name}_{kind}.png'))


## 5. 正式D 150000步


In [ ]:
assert SMOKE['iteration']==200
FORMAL_OUT=RESULTS/'formal_D'
run(train_command(FORMAL_OUT,ITERATIONS,'on'),cwd=CODE)
FORMAL=verify(FORMAL_OUT,ITERATIONS,True)
with (RESULTS/'workflow_verified.json').open('x') as f:
    json.dump({'code_ref':CODE_REF,'smoke':SMOKE,'formal':FORMAL},f,indent=2)
print('D completed and verified:',RESULTS)
print((FORMAL_OUT/'experiment_summary.md').read_text())


## 6. 全部写入Drive后释放GPU


In [ ]:
DISCONNECT_WHEN_VERIFIED=True
if DISCONNECT_WHEN_VERIFIED:
    done=json.loads((FORMAL_OUT/'completed.json').read_text())
    assert done['iteration']==150000 and done['final_test_complete'] is True
    assert (RESULTS/'workflow_verified.json').is_file()
    with (RESULTS/'runtime_release_requested.json').open('x') as f:
        json.dump({'formal_completed':150000,'final_test_complete':True,'results':str(RESULTS)},f)
    os.sync()
    from google.colab import runtime
    runtime.unassign()
